# 🧠 TACO-Net — Classification 3D Distribuée
## Graph CNN · Apprentissage Fédéré · Distillation de Connaissances
### ShapeNet via CAP3D — Notebook Kaggle complet

**Contenu du notebook :**
1. Installation & configuration
2. Téléchargement du sous-ensemble ShapeNet
3. Pipeline de données (.ply → tenseurs PyTorch)
4. Architecture **TACO-Net** (Graph CNN dynamique avec attention)
5. **Partie 1** — Baseline Centralisée (C1) + courbes de convergence
6. **Partie 2A** — Fédération Horizontale IID (C2)
7. **Partie 2A** — Fédération Horizontale Non-IID (C3)
8. **Partie 2B** — Fédération Verticale VFL (XYZ / RGB)
9. **Partie 3** — Distillation de Connaissances + Ablation α × τ (C4)
10. Comparaison finale C1/C2/C3/C4

> ⚙️ **Avant de lancer** : activer le GPU dans *Settings → Accelerator → GPU T4*


## ⚙️ 1. Installation & imports

In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])

pip("plyfile")
pip("huggingface_hub")

import os, random, copy, json, time, zipfile, warnings
from pathlib   import Path
from collections import defaultdict

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from torch.utils.data import Dataset, DataLoader, Subset
from plyfile import PlyData

warnings.filterwarnings("ignore")

# ── Reproductibilité ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Device   : {DEVICE}")


## 📦 2. Téléchargement du sous-ensemble ShapeNet (~3 Go)

In [ ]:
# ════════════════════════════════════════════════════════════
# OPTION A (recommandée) : ShapeNet déjà disponible sur Kaggle
# Ajoute ce dataset à ton notebook :
#   "+ Add Data" → cherche "shapenet" → ajoute "ShapeNet Point Clouds"
#   ou utilise le dataset kaggle : "mitkgarg/shapenet-part-dataset"
#
# OPTION B : Générer des données synthétiques (mode démo/test rapide)
# Active SYNTHETIC = True si tu n'as pas accès à ShapeNet sur Kaggle
# ════════════════════════════════════════════════════════════

SYNTHETIC = True   # ← met False si tu as ajouté ShapeNet via "+ Add Data"

PLY_DIR  = Path("/kaggle/working/ply_files")
PLY_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = "/kaggle/working/labeled_dataset.csv"

# ── Mapping synset ────────────────────────────────────────────
SYNSETS = {
    "03001627": "chair",    "04379243": "table",
    "02933112": "cabinet",  "02818832": "bed",
    "02958343": "car",      "02691156": "airplane",
    "04530566": "watercraft","02924116": "bus",
    "03211117": "display",  "04401088": "telephone",
    "03642806": "laptop",
    "03636649": "lamp",     "03797390": "mug",
    "04099429": "rifle",    "03467517": "guitar",
}

# ════════════════════════════════════════════════════════════
# OPTION A : Lire depuis un dataset Kaggle ShapeNet existant
# ════════════════════════════════════════════════════════════
def load_from_kaggle_shapenet():
    """
    Si tu as ajouté un dataset ShapeNet via '+ Add Data',
    adapte KAGGLE_PLY_ROOT au bon chemin de montage.
    Exemples de datasets disponibles :
      - /kaggle/input/shapenet-part-dataset/
      - /kaggle/input/shapenet55/
    """
    import glob
    KAGGLE_PLY_ROOT = "/kaggle/input/shapenet-part-dataset"
    if not os.path.exists(KAGGLE_PLY_ROOT):
        print("  ⚠️  Chemin ShapeNet non trouvé. Active SYNTHETIC=True ou ajuste KAGGLE_PLY_ROOT.")
        return False

    records = []
    for sid, lbl in SYNSETS.items():
        plys = glob.glob(f"{KAGGLE_PLY_ROOT}/**/{sid}*.ply", recursive=True)
        for p in plys[:200]:   # max 200 par classe
            dest = PLY_DIR / Path(p).name
            if not dest.exists():
                import shutil; shutil.copy(p, dest)
            records.append({"id": Path(p).stem, "label": lbl})

    if records:
        pd.DataFrame(records).to_csv(CSV_PATH, index=False)
        print(f"  ✓ {len(records)} objets chargés depuis Kaggle ShapeNet")
        return True
    return False

# ════════════════════════════════════════════════════════════
# OPTION B : Données synthétiques (sphères, cubes, cylindres…)
# Permet de tester TOUT le pipeline sans dataset réel
# ════════════════════════════════════════════════════════════
def generate_synthetic_ply(path, shape_type, n_pts=2048):
    """Génère un objet 3D synthétique et le sauvegarde en .ply."""
    from plyfile import PlyData, PlyElement
    import numpy as np

    rng = np.random.default_rng(abs(hash(str(path))) % (2**31))

    if shape_type == "sphere":
        theta = rng.uniform(0, 2*np.pi, n_pts)
        phi   = np.arccos(rng.uniform(-1, 1, n_pts))
        r     = rng.uniform(0.8, 1.0, n_pts)
        xyz   = np.stack([r*np.sin(phi)*np.cos(theta),
                          r*np.sin(phi)*np.sin(theta),
                          r*np.cos(phi)], axis=-1).astype(np.float32)
        rgb   = np.tile([0.8, 0.3, 0.3], (n_pts,1)).astype(np.float32)

    elif shape_type == "cube":
        face  = rng.integers(0, 6, n_pts)
        pts   = rng.uniform(-1, 1, (n_pts, 3)).astype(np.float32)
        for i,ax in enumerate([0,0,1,1,2,2]):
            mask = face==i; pts[mask,ax] = 1.0 if i%2==0 else -1.0
        xyz = pts
        rgb = np.tile([0.3, 0.6, 0.9], (n_pts,1)).astype(np.float32)

    elif shape_type == "cylinder":
        theta = rng.uniform(0, 2*np.pi, n_pts)
        h     = rng.uniform(-1, 1, n_pts)
        r     = rng.choice([1.0]*int(n_pts*.7) + list(rng.uniform(0,1,int(n_pts*.3))))
        xyz   = np.stack([r*np.cos(theta), h,
                          r*np.sin(theta)], axis=-1).astype(np.float32)
        rgb   = np.tile([0.3, 0.8, 0.4], (n_pts,1)).astype(np.float32)

    elif shape_type == "cone":
        h     = rng.uniform(0, 2, n_pts)
        theta = rng.uniform(0, 2*np.pi, n_pts)
        r     = (1 - h/2) * rng.uniform(0.8,1.0,n_pts)
        xyz   = np.stack([r*np.cos(theta), h-1,
                          r*np.sin(theta)], axis=-1).astype(np.float32)
        rgb   = np.tile([0.9, 0.7, 0.2], (n_pts,1)).astype(np.float32)

    else:  # random blob
        xyz = rng.standard_normal((n_pts, 3)).astype(np.float32)
        xyz /= np.linalg.norm(xyz,axis=1,keepdims=True).clip(1e-6)
        xyz *= rng.uniform(0.5,1.0,(n_pts,1)).astype(np.float32)
        rgb = rng.uniform(0,1,(n_pts,3)).astype(np.float32)

    # Ajout de bruit léger
    xyz += rng.normal(0, 0.02, xyz.shape).astype(np.float32)

    # Normalisation couleurs
    rgb = (rgb * 255).clip(0,255).astype(np.uint8)

    vertex = np.array(
        list(zip(xyz[:,0],xyz[:,1],xyz[:,2],
                 rgb[:,0],rgb[:,1],rgb[:,2])),
        dtype=[("x","f4"),("y","f4"),("z","f4"),
               ("red","u1"),("green","u1"),("blue","u1")]
    )
    PlyData([PlyElement.describe(vertex,"vertex")], text=False).write(str(path))


def generate_synthetic_dataset(n_per_class=120):
    """
    Génère n_per_class objets synthétiques par classe.
    Chaque classe est associée à une forme géométrique distincte.
    """
    # Chaque classe → forme géométrique (proxy visuel)
    CLASS_SHAPE = {
        "chair": "cube",      "table": "cube",
        "cabinet": "cube",    "bed": "cube",
        "car": "cube",        "airplane": "cylinder",
        "watercraft": "cylinder","bus": "cube",
        "display": "cube",    "telephone": "cylinder",
        "laptop": "cube",     "lamp": "cone",
        "mug": "cylinder",    "rifle": "cylinder",
        "guitar": "cylinder",
    }

    records = []
    total   = len(SYNSETS) * n_per_class
    done    = 0

    print(f"  Génération de {total} objets synthétiques...")
    for sid, lbl in SYNSETS.items():
        shape = CLASS_SHAPE.get(lbl, "sphere")
        for i in range(n_per_class):
            obj_id = f"{sid}_{i:05d}"
            ply_p  = PLY_DIR / f"{obj_id}.ply"
            if not ply_p.exists():
                generate_synthetic_ply(ply_p, shape)
            records.append({"id": obj_id, "label": lbl})
            done += 1
            if done % 100 == 0:
                print(f"    {done}/{total} ({lbl})", end="\r", flush=True)

    df = pd.DataFrame(records)
    df.to_csv(CSV_PATH, index=False)
    print(f"\n  ✓ Dataset synthétique prêt : {len(df)} objets, {df['label'].nunique()} classes")
    print(df["label"].value_counts().to_string())
    return df

# ════════════════════════════════════════════════════════════
# EXÉCUTION
# ════════════════════════════════════════════════════════════
if not SYNTHETIC:
    ok = load_from_kaggle_shapenet()
    if not ok:
        print("  → Bascule sur données synthétiques")
        generate_synthetic_dataset(n_per_class=120)
else:
    generate_synthetic_dataset(n_per_class=120)

print(f"\nFichiers .ply : {len(list(PLY_DIR.glob('*.ply')))}")


## 🗂️ 3. Construction du labeled_dataset.csv

In [ ]:
# ── Construction du labeled_dataset.csv (déjà fait en cellule 2) ──
# Vérification et affichage

df_labels = pd.read_csv(CSV_PATH)
print(f"Total objets : {len(df_labels)}")
print(f"Classes      : {df_labels['label'].nunique()}")
print()
print(df_labels["label"].value_counts().to_string())


## 🔧 4. Pipeline de données

In [ ]:
# ── Classes & mapping ────────────────────────────────────────
CLIENT_CLASSES = {
    0: ["chair","table","cabinet","bed"],
    1: ["car","airplane","watercraft","bus"],
    2: ["display","telephone","laptop"],
    3: ["lamp","mug","rifle","guitar"],
}
ALL_CLASSES = [c for v in CLIENT_CLASSES.values() for c in v]
CLASS2IDX   = {c:i for i,c in enumerate(ALL_CLASSES)}
IDX2CLASS   = {i:c for c,i in CLASS2IDX.items()}
NUM_CLASSES = len(CLASS2IDX)
print(f"Classes ({NUM_CLASSES}) :", ALL_CLASSES)

# ── Utilitaires ──────────────────────────────────────────────
def load_ply(path):
    v   = PlyData.read(path)["vertex"]
    xyz = np.stack([v["x"],v["y"],v["z"]],-1).astype(np.float32)
    rgb = (np.stack([v["red"],v["green"],v["blue"]],-1).astype(np.float32)/255.
           if "red" in v.data.dtype.names else np.zeros_like(xyz))
    return np.concatenate([xyz,rgb],-1)

def fps(pts, n):
    N = len(pts)
    if N <= n:
        return pts[np.concatenate([np.arange(N),
                                   np.random.choice(N, n-N)])]
    sel=[0]; dist=np.full(N,np.inf)
    for _ in range(n-1):
        d=((pts-pts[sel[-1]])**2).sum(-1); dist=np.minimum(dist,d)
        sel.append(int(np.argmax(dist)))
    return pts[np.array(sel)]

def normalize(xyz):
    xyz=xyz-xyz.mean(0)
    s=np.max(np.linalg.norm(xyz,axis=1))
    return xyz/s if s>0 else xyz

def augment(xyz):
    t=np.random.uniform(0,2*np.pi)
    R=np.array([[np.cos(t),-np.sin(t),0],
                [np.sin(t), np.cos(t),0],[0,0,1]],np.float32)
    xyz=xyz@R.T + np.random.normal(0,.01,xyz.shape).astype(np.float32)
    if random.random()<.5: xyz[:,0]*=-1
    return xyz

# ── Dataset ──────────────────────────────────────────────────
class ShapeNetDS(Dataset):
    def __init__(self, csv, ply_dir, n_pts=1024,
                 split="train", client_id=None, seed=42):
        self.ply_dir=Path(ply_dir); self.n_pts=n_pts; self.split=split
        df=pd.read_csv(csv)
        df=df[df["label"].isin(CLASS2IDX)].reset_index(drop=True)
        if client_id is not None:
            df=df[df["label"].isin(CLIENT_CLASSES[client_id])].reset_index(drop=True)
        rng=np.random.default_rng(seed); idx=rng.permutation(len(df))
        n=len(df); t=int(.7*n); v=int(.85*n)
        slc={"train":idx[:t],"val":idx[t:v],"test":idx[v:]}
        self.samples=df.iloc[slc[split]][["id","label"]].values.tolist()

    def __len__(self): return len(self.samples)

    def __getitem__(self,i):
        oid,lbl=self.samples[i]
        raw=load_ply(self.ply_dir/f"{oid}.ply")
        pts=fps(raw,self.n_pts)
        xyz=normalize(pts[:,:3]); rgb=pts[:,3:].clip(0,1)
        if self.split=="train": xyz=augment(xyz)
        return (torch.from_numpy(xyz), torch.from_numpy(rgb),
                torch.tensor(CLASS2IDX[lbl],dtype=torch.long))

def get_loaders(csv, ply_dir, n_pts=1024, batch=32,
                client_id=None, seed=42, nw=2):
    kw=dict(csv=csv,ply_dir=ply_dir,n_pts=n_pts,
            client_id=client_id,seed=seed)
    ds={s:ShapeNetDS(split=s,**kw) for s in ["train","val","test"]}
    return (DataLoader(ds["train"],batch,shuffle=True, num_workers=nw,pin_memory=True),
            DataLoader(ds["val"],  batch,shuffle=False,num_workers=nw,pin_memory=True),
            DataLoader(ds["test"], batch,shuffle=False,num_workers=nw,pin_memory=True))

# ── Test rapide ───────────────────────────────────────────────
TR,VA,TE = get_loaders(CSV_PATH, PLY_DIR, n_pts=1024, batch=32)
xyz_b,rgb_b,y_b = next(iter(TR))
print(f"Train {len(TR.dataset)} | Val {len(VA.dataset)} | Test {len(TE.dataset)}")
print(f"Batch → xyz:{xyz_b.shape}  rgb:{rgb_b.shape}  y:{y_b.shape}")


## 🕸️ 5. Architecture TACO-Net (Graph CNN)

**TACO-Net** = *Topology-Aware Convolutional Graph Network*

| Composant | Rôle |
|-----------|------|
| `knn_graph` | Reconstruit le graphe kNN **dynamiquement** sur les features courantes |
| `get_edge_features` | Calcule `[xi \|\| xj − xi]` pour chaque arête |
| `TACOConv` | EdgeConv + **attention par arête** (softmax sur k voisins) + résidu |
| Pooling global | `max + mean` concatenés → vecteur fixe |
| MLP classifieur | BatchNorm · LeakyReLU · Dropout |


In [ ]:
# ════════════════════════════════════════════════════════════
# Utilitaires graphe
# ════════════════════════════════════════════════════════════
def knn_graph(x, k):
    """(B,N,C) → (B,N,k) indices des k plus proches voisins (feature-space)."""
    B,N,C=x.shape
    xx=(x**2).sum(-1,keepdim=True)
    dist=(xx+xx.transpose(1,2)-2*x.bmm(x.transpose(1,2))).clamp(min=0)
    return dist.topk(k+1,dim=-1,largest=False).indices[:,:,1:]

def get_edge_features(x, idx):
    """(B,N,C),(B,N,k) → (B,N,k,2C)  features d'arêtes [xi || xj-xi]."""
    B,N,C=x.shape; k=idx.shape[-1]
    off   = torch.arange(B,device=x.device).view(B,1,1)*N
    xnb   = x.reshape(B*N,C)[(idx+off).reshape(B*N*k)].reshape(B,N,k,C)
    xctr  = x.unsqueeze(2).expand_as(xnb)
    return torch.cat([xctr, xnb-xctr],-1)

# ════════════════════════════════════════════════════════════
# Bloc TACO avec attention topologique
# ════════════════════════════════════════════════════════════
class TACOConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=20):
        super().__init__()
        self.k=k
        self.mlp=nn.Sequential(
            nn.Linear(2*in_ch,out_ch,bias=False),nn.BatchNorm1d(out_ch),nn.LeakyReLU(.2,True),
            nn.Linear(out_ch,  out_ch,bias=False),nn.BatchNorm1d(out_ch),nn.LeakyReLU(.2,True),
        )
        self.attn=nn.Linear(2*in_ch,1)
        self.sc  =nn.Linear(in_ch,out_ch,bias=False) if in_ch!=out_ch else nn.Identity()

    def forward(self,x):
        B,N,C=x.shape; k=self.k
        ef  = get_edge_features(x, knn_graph(x,k))          # (B,N,k,2C)
        aw  = F.softmax(self.attn(ef),dim=2)                 # (B,N,k,1)
        h   = self.mlp(ef.reshape(B*N*k,2*C)).reshape(B,N,k,-1)
        out = (h*aw).sum(2) + h.max(2).values                # att.sum + max
        return out + self.sc(x)

# ════════════════════════════════════════════════════════════
# TACO-Net complet (teacher / baseline)
# ════════════════════════════════════════════════════════════
class TACONet(nn.Module):
    def __init__(self,nc=15,k=20,dim=64,drop=.5,color=True):
        super().__init__()
        self.color=color; ic=6 if color else 3
        self.c1=TACOConv(ic,    dim,   k)
        self.c2=TACOConv(dim,   dim*2, k)
        self.c3=TACOConv(dim*2, dim*4, k)
        pd=dim*4*2
        self.clf=nn.Sequential(
            nn.Linear(pd,256),nn.BatchNorm1d(256),nn.LeakyReLU(.2,True),nn.Dropout(drop),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.LeakyReLU(.2,True),nn.Dropout(drop/2),
            nn.Linear(128,nc),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight,a=.2)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(self,xyz,rgb):
        x=torch.cat([xyz,rgb],-1) if self.color else xyz
        x=self.c3(self.c2(self.c1(x)))
        return torch.cat([x.max(1).values,x.mean(1)],-1)

    def forward(self,xyz,rgb): return self.clf(self.encode(xyz,rgb))
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ════════════════════════════════════════════════════════════
# TACONetSmall (student distillation, ≥4× moins de params)
# ════════════════════════════════════════════════════════════
class TACONetSmall(nn.Module):
    def __init__(self,nc=15,k=16,dim=32,drop=.4):
        super().__init__()
        self.c1=TACOConv(6,dim,  k)
        self.c2=TACOConv(dim,dim*2,k)
        pd=dim*2*2
        self.clf=nn.Sequential(
            nn.Linear(pd,64),nn.BatchNorm1d(64),nn.LeakyReLU(.2,True),nn.Dropout(drop),
            nn.Linear(64,nc),
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight,a=.2)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(self,xyz,rgb):
        x=self.c2(self.c1(torch.cat([xyz,rgb],-1)))
        return torch.cat([x.max(1).values,x.mean(1)],-1)

    def forward(self,xyz,rgb): return self.clf(self.encode(xyz,rgb))
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ── Vérification ─────────────────────────────────────────────
big   = TACONet(NUM_CLASSES)
small = TACONetSmall(NUM_CLASSES)
print(f"TACONet      : {big.n_params():>10,} paramètres")
print(f"TACONetSmall : {small.n_params():>10,} paramètres")
print(f"Ratio        : ×{big.n_params()/small.n_params():.1f}")
with torch.no_grad():
    out=big(xyz_b,rgb_b)
print(f"Forward OK   : {out.shape}  ✓")


## 🛠️ 6. Utilitaires : entraînement, évaluation, courbes de convergence

In [ ]:
# ── Classe History : courbes de convergence Graph CNN ─────────
class History:
    def __init__(self,label=""):
        self.label=label
        self.epochs=[]; self.tl=[]; self.vl=[]; self.ta=[]; self.va=[]

    def rec(self,ep,tl,vl,ta,va):
        self.epochs.append(ep);self.tl.append(tl);self.vl.append(vl)
        self.ta.append(ta);self.va.append(va)

    def plot(self,title=None,save=None):
        fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
        fig.suptitle(title or f"Convergence — {self.label}",fontsize=13,fontweight="bold")
        # Loss
        ax1.plot(self.epochs,self.tl,"b-o",ms=3,lw=1.8,label="Train")
        ax1.plot(self.epochs,self.vl,"r-s",ms=3,lw=1.8,label="Val")
        ax1.set(xlabel="Époque",ylabel="Loss",title="Cross-Entropy Loss")
        ax1.legend(); ax1.grid(alpha=.3)
        # Accuracy
        ax2.plot(self.epochs,[a*100 for a in self.ta],"b-o",ms=3,lw=1.8,label="Train")
        ax2.plot(self.epochs,[a*100 for a in self.va],"r-s",ms=3,lw=1.8,label="Val")
        ax2.set(xlabel="Époque",ylabel="Accuracy (%)",title="Accuracy",ylim=(0,100))
        ax2.legend(); ax2.grid(alpha=.3)
        plt.tight_layout()
        if save: plt.savefig(save,dpi=130,bbox_inches="tight")
        plt.show()

# ── Évaluation ────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model,loader,crit=None):
    model.eval()
    if crit is None: crit=nn.CrossEntropyLoss()
    tl=co=tot=0
    cc=defaultdict(int); ct=defaultdict(int)
    for xyz,rgb,y in loader:
        xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
        lg=model(xyz,rgb); tl+=crit(lg,y).item()*y.size(0)
        pr=lg.argmax(-1); co+=(pr==y).sum().item(); tot+=y.size(0)
        for c,p in zip(y.cpu().numpy(),pr.cpu().numpy()):
            ct[int(c)]+=1; cc[int(c)]+=int(c==p)
    per={c:cc[c]/ct[c] for c in ct}
    return tl/tot, co/tot, np.mean(list(per.values())), per

# ── Boucle d'entraînement ─────────────────────────────────────
def train_model(model, tr_ld, va_ld, epochs=80, lr=1e-3, wd=1e-4,
                label="model", patience=15):
    model=model.to(DEVICE)
    crit =nn.CrossEntropyLoss(label_smoothing=.1)
    opt  =optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
    sch  =optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-5)
    hist =History(label)
    best=0.; bs=None; pat=0

    print(f"\n{'═'*58}")
    print(f"  {label}  |  {model.n_params():,} params  |  {DEVICE}")
    print(f"{'═'*58}")
    print(f"  {'Ep':>4} | {'TrLoss':>8} | {'VaLoss':>8} | {'TrAcc':>7} | {'VaAcc':>7}")
    print(f"  {'-'*50}")

    for ep in range(1,epochs+1):
        model.train(); tl=co=tot=0
        for xyz,rgb,y in tr_ld:
            xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            loss=crit(model(xyz,rgb),y); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step()
            tl+=loss.item()*y.size(0); tot+=y.size(0)
            with torch.no_grad():
                co+=(model(xyz,rgb).argmax(-1)==y).sum().item()
        sch.step()
        tr_l=tl/tot; tr_a=co/tot
        va_l,va_a,_,_=evaluate(model,va_ld,crit)
        hist.rec(ep,tr_l,va_l,tr_a,va_a)
        if ep%10==0 or ep==1:
            print(f"  {ep:>4} | {tr_l:>8.4f} | {va_l:>8.4f} | {tr_a*100:>6.1f}% | {va_a*100:>6.1f}%")
        if va_a>best: best=va_a; bs=copy.deepcopy(model.state_dict()); pat=0
        else:
            pat+=1
            if pat>=patience: print(f"  [Early stop ep={ep}]"); break

    model.load_state_dict(bs)
    print(f"  → Meilleure Val Acc : {best*100:.2f}%")
    return model, hist


## 🏋️ 7. PARTIE 1 — Baseline Centralisée (C1)

In [ ]:
print("\n"+"█"*58)
print("  PARTIE 1 — BASELINE CENTRALISÉE (C1)")
print("  Toutes les données · TACONet complet")
print("█"*58)

model_c1, hist_c1 = train_model(
    TACONet(NUM_CLASSES), TR, VA,
    epochs=100, lr=1e-3, label="C1_Baseline", patience=20
)

# ── Courbes de convergence Graph CNN ──────────────────────────
hist_c1.plot(
    title="TACO-Net — C1 Baseline Centralisée\nCourbes de Convergence Graph CNN",
    save="/kaggle/working/C1_convergence.png"
)

# ── Évaluation test ───────────────────────────────────────────
_, c1_acc, c1_mcls, c1_per = evaluate(model_c1, TE)
print(f"  C1 Test Accuracy       : {c1_acc*100:.2f}%")
print(f"  C1 Mean Class Accuracy : {c1_mcls*100:.2f}%")
print(f"  C1 Paramètres          : {model_c1.n_params():,}")

# Sauvegarde du teacher pour la distillation
torch.save(model_c1.state_dict(), "/kaggle/working/C1_teacher.pt")
print("  ✓ Modèle sauvegardé → /kaggle/working/C1_teacher.pt")


## ⚖️ 8. FedAvg — implémentation manuelle (RÈGLE ABSOLUE : pas de lib tierce)

In [ ]:
# ════════════════════════════════════════════════════════════
# FedAvg implémenté CLÉ PAR CLÉ dans le state_dict PyTorch
# w(t+1) = Σ_k (n_k / n) * w_k^(t)
# ════════════════════════════════════════════════════════════

def fedavg(global_sd, client_sds, client_ns):
    """
    Agrégation FedAvg pondérée par taille de dataset local.
    Flottants  → moyenne pondérée.
    Entiers BN → copie du client dominant.
    """
    n_total  = sum(client_ns)
    weights  = [n/n_total for n in client_ns]
    dominant = int(np.argmax(client_ns))
    new_sd   = {}
    for key in global_sd:
        ref = client_sds[0][key]
        if ref.dtype.is_floating_point:
            agg = torch.zeros_like(ref, dtype=torch.float64)
            for w, sd in zip(weights, client_sds):
                agg += w * sd[key].double()
            new_sd[key] = agg.to(ref.dtype)
        else:
            new_sd[key] = client_sds[dominant][key].clone()   # buffers BN
    return new_sd


def local_train(global_sd, loader, local_ep, lr, wd):
    """Entraîne un client local. Retourne (state_dict, n_samples)."""
    m = TACONet(NUM_CLASSES)
    m.load_state_dict(copy.deepcopy(global_sd))   # copy.deepcopy OBLIGATOIRE
    m = m.to(DEVICE); m.train()
    opt  = optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=.1)
    for _ in range(local_ep):
        for xyz,rgb,y in loader:
            xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            loss=crit(m(xyz,rgb),y); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(),1.); opt.step()
    return m.state_dict(), len(loader.dataset)


def federated_loop(client_loaders, test_loader,
                   num_rounds=25, local_ep=5,
                   lr=5e-4, wd=1e-4, label="Fed"):
    """Boucle de fédération commune IID / Non-IID."""
    global_m  = TACONet(NUM_CLASSES)
    global_sd = global_m.state_dict()
    crit      = nn.CrossEntropyLoss()
    r_accs=[]; r_mcls=[]

    print(f"\n  {'Ronde':>6} | {'Acc':>8} | {'MeanCls':>8}")
    print(f"  {'-'*28}")

    for rnd in range(1, num_rounds+1):
        c_sds=[]; c_ns=[]
        for ld in client_loaders:
            sd,n = local_train(global_sd, ld, local_ep, lr, wd)
            c_sds.append(sd); c_ns.append(n)

        global_sd = fedavg(global_sd, c_sds, c_ns)    # ← FedAvg manuel
        global_m.load_state_dict(global_sd)

        _,acc,mcls,_ = evaluate(global_m, test_loader, crit)
        r_accs.append(acc); r_mcls.append(mcls)

        if rnd%5==0 or rnd==1:
            print(f"  {rnd:>6} | {acc*100:>7.2f}% | {mcls*100:>7.2f}%")

    return global_m, global_sd, r_accs, r_mcls


## 🌍 9. PARTIE 2A — HFL IID (C2)

In [ ]:
print("\n"+"█"*58)
print("  PARTIE 2A — HFL IID (C2)")
print("  4 clients · partition aléatoire uniforme")
print("█"*58)

# Partition IID : shuffle puis 4 parts égales
full_tr = ShapeNetDS(CSV_PATH, PLY_DIR, n_pts=1024, split="train")
parts   = np.array_split(np.random.default_rng(SEED).permutation(len(full_tr)), 4)
loaders_iid = [
    DataLoader(Subset(full_tr,list(p)), batch_size=16,
               shuffle=True, num_workers=2)
    for p in parts
]
for i,l in enumerate(loaders_iid):
    print(f"  Client IID {i} : {len(l.dataset)} samples")

model_c2, sd_c2, accs_c2, mcls_c2 = federated_loop(
    loaders_iid, TE, num_rounds=25, local_ep=5,
    lr=5e-4, label="C2_HFL_IID"
)

# Courbe accuracy par ronde
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(range(1,len(accs_c2)+1),[a*100 for a in accs_c2],
        "g-o",ms=4,lw=2,label="Val Accuracy")
ax.set(xlabel="Ronde de fédération",ylabel="Accuracy (%)",
       title="TACO-Net — C2 HFL IID\nAccuracy globale par ronde",ylim=(0,100))
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/kaggle/working/C2_rounds.png",dpi=130); plt.show()

_,c2_acc,c2_mcls,_ = evaluate(model_c2, TE)
print(f"  C2 Test Accuracy       : {c2_acc*100:.2f}%")
print(f"  C2 Mean Class Accuracy : {c2_mcls*100:.2f}%")


## 🗂️ 10. PARTIE 2A — HFL Non-IID (C3)

In [ ]:
print("\n"+"█"*58)
print("  PARTIE 2A — HFL NON-IID (C3)")
print("  4 clients · partition sémantique par domaine")
print("█"*58)

loaders_niid = []
for cid in range(4):
    ds_c = ShapeNetDS(CSV_PATH, PLY_DIR, n_pts=1024,
                      split="train", client_id=cid)
    if len(ds_c)==0:
        print(f"  ⚠️  Client {cid} vide → fallback IID")
        loaders_niid.append(loaders_iid[cid])
    else:
        loaders_niid.append(
            DataLoader(ds_c,batch_size=16,shuffle=True,num_workers=2))
    print(f"  Client {cid} {list(CLIENT_CLASSES[cid])[:2]}... "
          f": {len(loaders_niid[-1].dataset)} samples")

model_c3, sd_c3, accs_c3, mcls_c3 = federated_loop(
    loaders_niid, TE, num_rounds=25, local_ep=5,
    lr=5e-4, label="C3_HFL_NonIID"
)

# Analyse hors-domaine
print("\n  Analyse performance hors-domaine :")
_,_,_,per_c3 = evaluate(model_c3, TE)
for cid in range(4):
    own   = [CLASS2IDX[c] for c in CLIENT_CLASSES[cid] if c in CLASS2IDX]
    other = [i for i in per_c3 if i not in own]
    ao = np.mean([per_c3[i] for i in own   if i in per_c3]) if own   else 0
    ax = np.mean([per_c3[i] for i in other if i in per_c3]) if other else 0
    print(f"  Client {cid} | propres: {ao*100:.1f}%  | hors-domaine: {ax*100:.1f}%")

# Courbe
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(range(1,len(accs_c3)+1),[a*100 for a in accs_c3],
        "r-o",ms=4,lw=2,label="Val Accuracy")
ax.set(xlabel="Ronde de fédération",ylabel="Accuracy (%)",
       title="TACO-Net — C3 HFL Non-IID\nAccuracy globale par ronde",ylim=(0,100))
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/kaggle/working/C3_rounds.png",dpi=130); plt.show()

_,c3_acc,c3_mcls,_ = evaluate(model_c3, TE)
print(f"  C3 Test Accuracy       : {c3_acc*100:.2f}%")
print(f"  C3 Mean Class Accuracy : {c3_mcls*100:.2f}%")


## 🔀 11. PARTIE 2B — Fédération Verticale VFL (XYZ / RGB)

In [ ]:
print("\n"+"█"*58)
print("  PARTIE 2B — FÉDÉRATION VERTICALE (VFL)")
print("  Entité A : XYZ uniquement")
print("  Entité B : RGB uniquement")
print("  Serveur  : fusionne [eA || eB] → classification")
print("█"*58)

# ── Encodeurs VFL ─────────────────────────────────────────────
class VFLEncoder(nn.Module):
    def __init__(self,in_ch=3,dim=64,k=20):
        super().__init__()
        self.c1=TACOConv(in_ch,dim,k); self.c2=TACOConv(dim,dim*2,k)
        self.out_dim=dim*2*2
    def forward(self,x):
        h=self.c2(self.c1(x))
        return torch.cat([h.max(1).values,h.mean(1)],-1)
    def n_params(self): return sum(p.numel() for p in self.parameters())

class VFLServer(nn.Module):
    def __init__(self,dA,dB,nc):
        super().__init__()
        self.f=nn.Sequential(
            nn.Linear(dA+dB,256),nn.BatchNorm1d(256),nn.ReLU(True),nn.Dropout(.4),
            nn.Linear(256,nc)
        )
    def forward(self,eA,eB): return self.f(torch.cat([eA,eB],-1))
    def n_params(self): return sum(p.numel() for p in self.parameters())

# ── Instanciation ─────────────────────────────────────────────
entA   = VFLEncoder(3,64,20).to(DEVICE)    # Entité A : géométrie XYZ
entB   = VFLEncoder(3,64,20).to(DEVICE)    # Entité B : couleurs RGB
srvVFL = VFLServer(entA.out_dim,entB.out_dim,NUM_CLASSES).to(DEVICE)

print(f"  Entité A (XYZ) : {entA.n_params():,} params")
print(f"  Entité B (RGB) : {entB.n_params():,} params")
print(f"  Serveur fusion : {srvVFL.n_params():,} params")
print()

# Optimiseurs SÉPARÉS — isolation des entités
optA   = optim.AdamW(entA.parameters(),  lr=1e-3,weight_decay=1e-4)
optB   = optim.AdamW(entB.parameters(),  lr=1e-3,weight_decay=1e-4)
optSrv = optim.AdamW(srvVFL.parameters(),lr=1e-3,weight_decay=1e-4)
crit   = nn.CrossEntropyLoss(label_smoothing=.1)
hist_vfl=History("VFL"); best_vfl=0.

for ep in range(1,61):
    entA.train(); entB.train(); srvVFL.train()
    tl=co=tot=0
    for xyz,rgb,y in TR:
        xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
        optA.zero_grad(); optB.zero_grad(); optSrv.zero_grad()

        # Encodage local : seuls les embeddings quittent chaque entité
        eA = entA(xyz)           # Entité A → embedding géométrie
        eB = entB(rgb)           # Entité B → embedding couleur

        # Fusion + prédiction côté serveur
        lg = srvVFL(eA, eB)
        loss = crit(lg,y); loss.backward()

        for op in [optA,optB,optSrv]:
            nn.utils.clip_grad_norm_(list(op.param_groups[0]['params']),1.)
            op.step()
        tl+=loss.item()*y.size(0); co+=(lg.argmax(-1)==y).sum().item(); tot+=y.size(0)

    # Validation
    entA.eval(); entB.eval(); srvVFL.eval()
    vc=vt=0
    with torch.no_grad():
        for xyz,rgb,y in VA:
            xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
            vc+=(srvVFL(entA(xyz),entB(rgb)).argmax(-1)==y).sum().item(); vt+=y.size(0)
    va=vc/vt
    hist_vfl.rec(ep,tl/tot,0,co/tot,va)
    if va>best_vfl: best_vfl=va
    if ep%10==0:
        print(f"  Ep {ep:>3} | Tr {co/tot*100:.1f}% | Val {va*100:.1f}%")

print(f"\n  VFL Meilleure Val.Acc : {best_vfl*100:.2f}%")
print("  ✓ Confidentialité : aucune donnée brute échangée")
print("    Entité A n'a JAMAIS vu les RGB")
print("    Entité B n'a JAMAIS vu les XYZ")

hist_vfl.plot(title="TACO-Net — Fédération Verticale VFL\nCourbes de convergence",
              save="/kaggle/working/VFL_convergence.png")


## 🎓 12. PARTIE 3 — Distillation de Connaissances (C4)

In [ ]:
print("\n"+"█"*58)
print("  PARTIE 3 — DISTILLATION DE CONNAISSANCES (C4)")
print("  Enseignant : TACONet  |  Élève : TACONetSmall")
print(f"  Ratio de compression : ×{model_c1.n_params()//TACONetSmall(NUM_CLASSES).n_params()}")
print("█"*58)

# ── Perte de distillation ─────────────────────────────────────
def distil_loss(zs, zt, y, alpha, tau):
    """
    L = (1-α)*CE(zs, y) + α*τ²*KL( σ(zt/τ) || σ(zs/τ) )
    Les logits zt de l'enseignant sont DÉTACHÉS du graphe.
    """
    ce   = F.cross_entropy(zs, y)
    soft_t = F.log_softmax(zt.detach()/tau, -1)   # détaché !
    soft_s = F.softmax(zs/tau, -1)
    kd   = F.kl_div(soft_t, soft_s, reduction="batchmean") * tau**2
    return (1-alpha)*ce + alpha*kd

# ── Entraînement student ──────────────────────────────────────
teacher_c4 = TACONet(NUM_CLASSES).to(DEVICE)
teacher_c4.load_state_dict(
    torch.load("/kaggle/working/C1_teacher.pt", map_location=DEVICE))
teacher_c4.eval()

def train_student(alpha, tau, epochs=70, label="student"):
    stu = TACONetSmall(NUM_CLASSES).to(DEVICE)
    opt = optim.AdamW(stu.parameters(),lr=1e-3,weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-5)
    hist=History(label); best=0.; bs=None; pat=0
    for ep in range(1,epochs+1):
        stu.train(); tl=co=tot=0
        for xyz,rgb,y in TR:
            xyz,rgb,y=xyz.to(DEVICE),rgb.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            zs=stu(xyz,rgb)
            with torch.no_grad(): zt=teacher_c4(xyz,rgb)
            loss=distil_loss(zs,zt,y,alpha,tau)
            loss.backward()
            nn.utils.clip_grad_norm_(stu.parameters(),1.); opt.step()
            tl+=loss.item()*y.size(0); co+=(zs.argmax(-1)==y).sum().item(); tot+=y.size(0)
        sch.step()
        vl,va,_,_=evaluate(stu,VA)
        hist.rec(ep,tl/tot,vl,co/tot,va)
        if va>best: best=va; bs=copy.deepcopy(stu.state_dict()); pat=0
        else:
            pat+=1
            if pat>=12: break
    stu.load_state_dict(bs); return stu,hist,best

# ── Ablation α × τ ────────────────────────────────────────────
ALPHAS=[0.3,0.5,0.7,0.9]; TAUS=[1.,2.,4.,8.]
grid=np.zeros((len(ALPHAS),len(TAUS)))

print("\n  Ablation α × τ en cours...")
for i,a in enumerate(ALPHAS):
    for j,t in enumerate(TAUS):
        _,_,acc=train_student(a,t,epochs=30,label=f"ab_a{a}_t{t}")
        grid[i,j]=acc*100
        print(f"    α={a}  τ={t}  →  {acc*100:.1f}%")

# Heatmap
fig,ax=plt.subplots(figsize=(8,5))
im=ax.imshow(grid,cmap="YlOrRd",aspect="auto",
             vmin=grid.min()-1,vmax=grid.max())
ax.set_xticks(range(len(TAUS)));   ax.set_xticklabels([f"τ={t}" for t in TAUS])
ax.set_yticks(range(len(ALPHAS))); ax.set_yticklabels([f"α={a}" for a in ALPHAS])
ax.set_title("Val Accuracy (%) — Ablation α × τ\nTACO-Net Distillation",fontweight="bold")
plt.colorbar(im,ax=ax,label="Accuracy (%)")
for i in range(len(ALPHAS)):
    for j in range(len(TAUS)):
        ax.text(j,i,f"{grid[i,j]:.1f}%",ha="center",va="center",
                fontsize=9,fontweight="bold",
                color="white" if grid[i,j]>grid.max()*.88 else "black")
plt.tight_layout()
plt.savefig("/kaggle/working/C4_ablation_heatmap.png",dpi=130); plt.show()

bi,bj=np.unravel_index(np.argmax(grid),grid.shape)
best_alpha,best_tau=ALPHAS[bi],TAUS[bj]
print(f"\n  Meilleur couple : α={best_alpha}  τ={best_tau}  →  {grid[bi,bj]:.1f}%")

# Entraînement final du student avec les meilleurs hyperparams
print("\n  Entraînement final du student...")
student_c4,hist_c4,_=train_student(best_alpha,best_tau,epochs=80,label="C4_Student")

hist_c4.plot(
    title=f"TACO-Net Distillation (C4)\nα={best_alpha}  τ={best_tau} — Courbes de convergence",
    save="/kaggle/working/C4_convergence.png"
)

_,c4_acc,c4_mcls,_=evaluate(student_c4,TE)
print(f"\n  C4 Test Accuracy       : {c4_acc*100:.2f}%")
print(f"  C4 Mean Class Accuracy : {c4_mcls*100:.2f}%")
print(f"  C4 Paramètres (élève)  : {student_c4.n_params():,}")
print(f"  Ratio compression      : ×{model_c1.n_params()/student_c4.n_params():.1f}")


## 📊 13. Comparaison finale — C1 / C2 / C3 / C4

In [ ]:
print("\n"+"█"*58)
print("  COMPARAISON FINALE — 4 CONFIGURATIONS")
print("█"*58)

CFGS = {
    "C1 Baseline"    :{"acc":c1_acc, "mcls":c1_mcls,"params":model_c1.n_params()},
    "C2 HFL IID"     :{"acc":c2_acc, "mcls":c2_mcls,"params":model_c2.n_params()},
    "C3 HFL Non-IID" :{"acc":c3_acc, "mcls":c3_mcls,"params":model_c3.n_params()},
    "C4 Distillation":{"acc":c4_acc, "mcls":c4_mcls,"params":student_c4.n_params()},
}
labels = list(CFGS.keys())
accs   = [v["acc"]*100  for v in CFGS.values()]
mcls_v = [v["mcls"]*100 for v in CFGS.values()]
params = [v["params"]/1e6 for v in CFGS.values()]
COLS   = ["#2196F3","#4CAF50","#FF9800","#9C27B0"]

fig=plt.figure(figsize=(18,12))
gs =gridspec.GridSpec(2,2,figure=fig,hspace=.4,wspace=.35)

def bar_chart(ax,vals,title,ylabel,ylim=None):
    bars=ax.bar(labels,vals,color=COLS,edgecolor="black",lw=.8)
    ax.set(ylabel=ylabel,title=title)
    if ylim: ax.set_ylim(ylim)
    ax.grid(axis="y",alpha=.3); ax.tick_params(axis="x",rotation=12)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+.3 if ylim else b.get_height()+.005,
                f"{v:.1f}{'%' if ylim else 'M'}",ha="center",fontweight="bold",fontsize=10)
    return bars

ax1=fig.add_subplot(gs[0,0])
bar_chart(ax1,accs,"Précision Globale (Test)","Accuracy (%)",(0,100))
ax1.axhline(accs[0],color="gray",ls="--",alpha=.5,label="Baseline ref")
ax1.legend()

ax2=fig.add_subplot(gs[0,1])
bar_chart(ax2,mcls_v,"Précision Moyenne par Classe","Mean Class Acc (%)",(0,100))

ax3=fig.add_subplot(gs[1,0])
bar_chart(ax3,params,"Taille du Modèle","Paramètres (M)")

# Courbes de convergence superposées
ax4=fig.add_subplot(gs[1,1])
ax4.plot(hist_c1.epochs,[a*100 for a in hist_c1.va],COLS[0],lw=2,label="C1 Baseline")
ax4.plot(range(1,len(accs_c2)+1),[a*100 for a in accs_c2],
         COLS[1],lw=2,ls="--",label="C2 HFL IID")
ax4.plot(range(1,len(accs_c3)+1),[a*100 for a in accs_c3],
         COLS[2],lw=2,ls="-.",label="C3 HFL Non-IID")
ax4.plot(hist_c4.epochs,[a*100 for a in hist_c4.va],
         COLS[3],lw=2,ls=":",label="C4 Distillation")
ax4.set(xlabel="Époque / Ronde",ylabel="Val Accuracy (%)",
        title="Convergence — 4 configurations superposées",ylim=(0,100))
ax4.legend(fontsize=9); ax4.grid(alpha=.3)

fig.suptitle("TACO-Net — Comparaison C1/C2/C3/C4 sur ShapeNet",
             fontsize=14,fontweight="bold")
plt.savefig("/kaggle/working/comparison_final.png",dpi=130,bbox_inches="tight")
plt.show()

# ── Tableau terminal ──────────────────────────────────────────
print(f"\n  {'CONFIG':<22} | {'TEST ACC':>9} | {'MEAN CLS':>9} | {'PARAMS':>12}")
print(f"  {'─'*60}")
for name,v in CFGS.items():
    flag="  ← MEILLEUR" if v["acc"]==max(d["acc"] for d in CFGS.values()) else ""
    print(f"  {name:<22} | {v['acc']*100:>8.2f}% | {v['mcls']*100:>8.2f}% | {v['params']:>12,}{flag}")

print()
print("  Fichiers générés :")
import glob
for f in sorted(glob.glob("/kaggle/working/*.png")+glob.glob("/kaggle/working/*.pt")):
    print(f"    {f}")
print("\n✅ Projet complet terminé !")
